# Part D3: Outlier Handling via Trimmed Reconstruction

Part D2 showed that iterative SVD handles missing data well. But what about **outliers** —
observations that are present but corrupted with large errors?

Standard iterative SVD treats all observed entries as ground truth, so outliers permanently
distort the rank-3 reconstruction. We introduce **trimmed reconstruction**: at each iteration,
identify the top q% largest-residual observed entries as suspected outliers and treat them as
missing, letting the rank-3 reconstruction overwrite them. This is controlled by the
`trim_fraction` parameter in `factorize_tomasi_kanade`.

In [22]:
import numpy as np
from src.data_loader import generate_sfm_data
from src.factorization import factorize_tomasi_kanade

In [23]:
NUM_POINTS = 50
NUM_FRAMES = 10
SIGMA = 0.01
SEED = 42

outlier_rates = [0.0, 0.02, 0.05, 0.10, 0.15, 0.20]
trim_fractions = [0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.10]

## Experiment : Fixed Outlier Rate, Varying Trim Fraction

We fix the outlier rate at 5% and sweep `trim_fraction` from 0% to 10%.
For each setting we report reprojection RMSE.

In [24]:
outlier_rate_fixed = 0.05
seeds = range(100)

print(f"{'trim_fraction':>14s}  {'Reproj RMSE':>12s}  {'Outliers ignored':>17s}  {'Inliers ignored':>16s}")
print("-" * 65)

for tf in trim_fractions:
    reproj_list = []
    outliers_ignored_list = []
    inliers_ignored_list = []
    for seed in seeds:
        data = generate_sfm_data(
            num_points=NUM_POINTS, num_frames=NUM_FRAMES,
            sigma=SIGMA, outlier_rate=outlier_rate_fixed, outlier_magnitude=5.0,
            seed=seed,
        )
        result = factorize_tomasi_kanade(
            data["measurement_matrix"], missing_strat="iterative-svd", trim_fraction=tf,
            outlier_mask=data["outlier_mask"],
        )
        m, s, t, hist, trim_stats = result
        reproj_list.append(hist[-1])
        outliers_ignored_list.append(trim_stats["outliers_ignored"])
        inliers_ignored_list.append(trim_stats["inliers_ignored"])
    print(f"{tf:>14.0%}  {np.mean(reproj_list):>12.6f}  {np.mean(outliers_ignored_list):>17.1f}  {np.mean(inliers_ignored_list):>16.1f}")

 trim_fraction   Reproj RMSE   Outliers ignored   Inliers ignored
-----------------------------------------------------------------
            0%      0.541755                0.0               0.0
            1%      0.391311               10.0               0.0
            2%      0.277339               19.9               0.1
            3%      0.188843               29.5               0.5
            4%      0.136065               37.5               2.5
            5%      0.110346               42.3               7.7
            6%      0.095290               44.4              15.7
            7%      0.087938               45.2              24.8
            8%      0.083428               45.6              34.4
            9%      0.079728               45.6              44.4
           10%      0.076506               46.0              54.0
